# Two-stage ViT fine-tuning on CIFAR-10

Fine-tuning ViT-B/16 (in21k) on a subset of CIFAR-10. First only the classifier head is trained, then the whole network with separate learning rates for the backbone and the head. I stopped the second stage after one epoch, at that point validation accuracy was 0.975.

In [1]:
import torch
import numpy as np
import pandas as pd
from PIL import Image
import torchvision.transforms.v2 as transforms
from torchvision import tv_tensors

In [2]:
from datasets import load_dataset

dataset_train = load_dataset(
    'uoft-cs/cifar10',  # Changed from 'cifar10'
    split="train"
)

dataset_test = load_dataset(
    'uoft-cs/cifar10',  # Changed from 'cifar10'
    split="test"
)

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/120M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/23.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [6]:
from transformers import ViTImageProcessor

# import model
model_id = 'google/vit-base-patch16-224-in21k'
feature_extractor = ViTImageProcessor.from_pretrained(
    model_id
)

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

In [14]:
from torch.utils.data import Dataset, DataLoader

class PIL_Dataset(Dataset):
    def __init__(self, pil_dataset, image_processor):
        self.dataset = pil_dataset
        self.processor = image_processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]

        pil_image = item['img']
        label = item['label']

        inputs = self.processor(images=pil_image, return_tensors="pt")
        pixel_values = inputs['pixel_values'][0]

        return {
            'pixel_values': pixel_values,
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_dataset = PIL_Dataset(dataset_train.select(range(8000)), feature_extractor)
test_dataset = PIL_Dataset(dataset_test.select(range(4000)), feature_extractor)
train_loader = DataLoader(train_dataset, batch_size = 64, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 64)

In [15]:
from transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(model_id, num_labels=10, ignore_mismatched_sizes=True)

model.to('cuda')

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermed

In [16]:
for idx, data in enumerate(train_loader):
  print(data['labels'].shape)
  break

torch.Size([64])


In [17]:
from torch.optim import AdamW
from transformers import get_scheduler

def train_one_epoch(model, dataloader, optimizer, scheduler=None):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for idx, data in enumerate(dataloader):
      #print(f"RuN {idx}")
      X_train = data['pixel_values']
      y_train = data['labels']
      X_train = X_train.to('cuda')
      y_train = y_train.to('cuda')

      optimizer.zero_grad()

      outputs = model(pixel_values=X_train, labels=y_train)
      loss = outputs.loss
      logits = outputs.logits

      loss.backward()

      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

      optimizer.step()

      if scheduler is not None:
        scheduler.step()

      total_loss += loss.item()
      preds = logits.argmax(-1)
      correct += (preds == y_train).sum().item()
      total += len(y_train)

    return total_loss / len(dataloader), correct / total

def evaluate(model, dataloader):
  model.eval()
  correct = 0
  total = 0
  with torch.no_grad():
      for batch in dataloader:
          X_val = batch["pixel_values"].to('cuda')
          y_val = batch["labels"].to('cuda')
          outputs = model(pixel_values=X_val)
          preds = outputs.logits.argmax(-1)
          correct += (preds == y_val).sum().item()
          total += y_val.size(0)
  return correct / total

# --- CONFIGURATION HYPERPARAMETERS ---
EPOCHS_PHASE_1 = 2   # Warm up the classification head
EPOCHS_PHASE_2 = 6   # Full fine-tuning
TOTAL_EPOCHS = EPOCHS_PHASE_1 + EPOCHS_PHASE_2

LR_BACKBONE = 2e-5   # Tiny learning rate for ViT encoder to preserve features
LR_HEAD = 1e-4       # Faster learning rate for the fresh linear head
WEIGHT_DECAY = 1e-2  # High weight decay works exceptionally well for Transformers

In [18]:
from transformers import get_scheduler
from torch.optim import AdamW
from tqdm import tqdm

for name, param in model.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False
    else:
        param.requires_grad = True

# Optimize ONLY the classifier layer
optim1 = AdamW(model.classifier.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)

print("Task 1")

for epoch in tqdm(range(EPOCHS_PHASE_1)):
    train_loss, train_acc = train_one_epoch(model, train_loader, optim1)
    print(f"[Phase 1 - Epoch {epoch+1}/{EPOCHS_PHASE_1}] Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")

print(f"PHASE 1 ACCURACY: {evaluate(model, test_loader)}")

print('Done 1')


Task 1


 50%|█████     | 1/2 [01:49<01:49, 109.75s/it]

[Phase 1 - Epoch 1/2] Loss: 2.0558 | Train Acc: 0.6431


100%|██████████| 2/2 [03:45<00:00, 112.89s/it]

[Phase 1 - Epoch 2/2] Loss: 1.6491 | Train Acc: 0.9219


PHASE 1 ACCURACY: 0.9375
Done 1


In [19]:
for param in model.parameters():
  param.requires_grad = True

optim2 = AdamW([
    {'params':model.vit.parameters(), 'lr' : LR_BACKBONE},
    {'params':model.classifier.parameters(), 'lr' : LR_HEAD}
], weight_decay = WEIGHT_DECAY)

total_steps = len(train_loader) * EPOCHS_PHASE_2
warmup_steps = int(total_steps * 1/10)

scheduler = get_scheduler(name = 'cosine', optimizer = optim2, num_warmup_steps = warmup_steps, num_training_steps = total_steps)

print('TASK 2')

best_val_acc = 0.0

for epoch in tqdm(range(EPOCHS_PHASE_2)):
    train_loss, train_acc = train_one_epoch(model, train_loader, optim2, scheduler)
    val_acc = evaluate(model, test_loader)
    print(f"[Phase 2 - Epoch {epoch+1}/{EPOCHS_PHASE_2}] Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc

print(f"\nTraining Complete! Highest Validation Accuracy: {best_val_acc:.4f}")

TASK 2


 17%|█▋        | 1/6 [05:51<29:18, 351.79s/it]

[Phase 2 - Epoch 1/6] Loss: 0.9465 | Train Acc: 0.9644 | Val Acc: 0.9752


 17%|█▋        | 1/6 [06:31<32:38, 391.72s/it]
